**1. Objective**

The objective is to develop an efficient and accurate model for breast cancer classification (malignant vs benign) by:

Applying Genetic Algorithm (GA) for optimal feature selection
Using Support Vector Machine (SVM) for classification
Reducing model complexity while maintaining high accuracy
Avoiding overfitting using cross-validation and regularization

**2. Dataset Used**

We used the Breast Cancer Wisconsin Dataset

Dataset Link:

https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html

Dataset Details:

Total samples: 569

Features: 30 numerical features

Classes:

0 → Malignant

1 → Benign

**3. Tools / Software Used**

Python (Google Colab)

Libraries:

NumPy

Scikit-learn

DEAP (Genetic Algorithm)

Matplotlib (optional for visualization)


**4. Methodology / Algorithm**

🔹 Step 1: Data Preprocessing
Loaded dataset using sklearn
Split data into:
Training set (70%)

Testing set (30%)

🔹 Step 2: Feature Selection using Genetic Algorithm
Each chromosome = binary vector of length 30

1 → feature selected

0 → feature ignored

Fitness Function:
Train SVM on selected features
Use 5-fold cross-validation

Fitness = mean accuracy

Fitness=Cross-validation Accuracy


🔹 Step 3: GA Operations
Selection → Tournament selection
Crossover → Two-point crossover
Mutation → Bit flip (probability = 0.05)

🔹 Step 4: Model Training
Train SVM with:
Linear kernel
Regularization parameter C=0.1

🔹 Step 5: Overfitting Control
Cross-validation used during GA
Regularization applied in SVM
Final evaluation done on unseen test data

🔹 Step 6: Final Evaluation
Accuracy
Precision
Recall
F1-score


**5. Output Obtained**

Final Results:

Selected Features: 18 (from GA)

Final Test Accuracy: 96.49%

Test Samples: 171



Classification Report

Class	        Precision	 Recall	 F1-score

Malignant (0)	0.97	     0.94	   0.95

Benign (1)	  0.96	     0.98	   0.97

**6. Performance Evaluation**

🔹 Metrics:

Metric	Value

Accuracy	96.49%

Precision	~96–97%

Recall	~94–98%

F1-score	~95–97%

 Key Observations:

High accuracy achieved with reduced features
No overfitting (unlike previous 100% result)
Model generalizes well on unseen data
Slight improvement needed for malignant detection (important in healthcare)

**7. Comparison with Existing Work**

Method	Accuracy

Decision Tree	~93%

KNN	~95%

SVM (without GA)	~96–97%

GA + SVM (Your Model)	96.49%

In [ ]:
import random
import numpy as np
from deap import base, creator, tools, algorithms
from sklearn import datasets
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
data = datasets.load_breast_cancer()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

num_features = X.shape[1]
print("Total features:", num_features)
print("Train samples:", X_train.shape[0], "| Test samples:", X_test.shape[0])

In [ ]:
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.attr_bool, n=num_features)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
def evalFitness(individual):
    selected = [i for i in range(len(individual)) if individual[i] == 1]
    if len(selected) == 0:
        return 0,
    X_selected = X[:, selected]
    model = SVC(kernel='linear', C=0.1)
    scores = cross_val_score(model, X_selected, y, cv=5)
    return scores.mean(),

toolbox.register("evaluate", evalFitness)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

In [ ]:
population = toolbox.population(n=20)

result, log = algorithms.eaSimple(
    population,
    toolbox,
    cxpb=0.5,
    mutpb=0.2,
    ngen=10,
    verbose=True
)

In [ ]:
best_ind = tools.selBest(population, k=1)[0]
selected_features = [i for i in range(len(best_ind)) if best_ind[i] == 1]

print("Selected Feature Indices:", selected_features)
print("Number of Features Selected:", len(selected_features))

In [ ]:
X_train_sel = X_train[:, selected_features]
X_test_sel  = X_test[:, selected_features]

model = SVC(kernel='linear', C=0.1)
model.fit(X_train_sel, y_train)

pred = model.predict(X_test_sel)

In [ ]:
print("Final Test Accuracy:", round(accuracy_score(y_test, pred) * 100, 2), "%")
print("\nClassification Report:\n", classification_report(y_test, pred,
      target_names=["Malignant (0)", "Benign (1)"]))

In [ ]:
final_scores = cross_val_score(model, X[:, selected_features], y, cv=5)
print("Cross-Validation Accuracy (5-fold):", round(final_scores.mean() * 100, 2), "%")